In [19]:
import pickle
import pandas as pd
import os

In [3]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

c:\Users\mokon\Documents\Anaconda\Lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\mokon\Documents\Anaconda\Lib\site-packages\sklearn\base.py:318: UserWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [5]:
def data(year, month):
    df = read_data(f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{int(year):04d}-{int(month):02d}.parquet')
    return df

df = data(2023, 3)

df.head()

In [6]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val) #The standard deviation of the prediction is: 6.24 (y_pred.std())

In [7]:
print("The standard devaition of the prediction is: {}".format(y_pred.std()))

The standard devaition of the prediction is: 6.247488852238703


In [10]:
df["year"] = df["tpep_pickup_datetime"].dt.year
df["month"] = df["tpep_pickup_datetime"].dt.month
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration,year,month
0,2,2023-03-01 00:06:43,2023-03-01 00:16:43,1.0,0.00,1.0,N,238,42,2,...,0.5,0.00,0.0,1.0,11.10,0.0,0.00,10.000000,2023,3
1,2,2023-03-01 00:08:25,2023-03-01 00:39:30,2.0,12.40,1.0,N,138,231,1,...,0.5,12.54,0.0,1.0,76.49,2.5,1.25,31.083333,2023,3
2,1,2023-03-01 00:15:04,2023-03-01 00:29:26,0.0,3.30,1.0,N,140,186,1,...,0.5,4.65,0.0,1.0,28.05,2.5,0.00,14.366667,2023,3
3,1,2023-03-01 00:49:37,2023-03-01 01:01:05,1.0,2.90,1.0,N,140,43,1,...,0.5,4.10,0.0,1.0,24.70,2.5,0.00,11.466667,2023,3
4,2,2023-03-01 00:08:04,2023-03-01 00:11:06,1.0,1.23,1.0,N,79,137,1,...,0.5,2.44,0.0,1.0,14.64,2.5,0.00,3.033333,2023,3


In [15]:
df = df.reset_index(drop=True)
df["ride_id"] = df.apply(
    lambda row: f"{row['year']:04d}/{row['month']:02d}_{row.name}", axis=1
)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration,year,month,ride_id
0,2,2023-03-01 00:06:43,2023-03-01 00:16:43,1.0,0.00,1.0,N,238,42,2,...,0.00,0.0,1.0,11.10,0.0,0.00,10.000000,2023,3,2023/03_0
1,2,2023-03-01 00:08:25,2023-03-01 00:39:30,2.0,12.40,1.0,N,138,231,1,...,12.54,0.0,1.0,76.49,2.5,1.25,31.083333,2023,3,2023/03_1
2,1,2023-03-01 00:15:04,2023-03-01 00:29:26,0.0,3.30,1.0,N,140,186,1,...,4.65,0.0,1.0,28.05,2.5,0.00,14.366667,2023,3,2023/03_2
3,1,2023-03-01 00:49:37,2023-03-01 01:01:05,1.0,2.90,1.0,N,140,43,1,...,4.10,0.0,1.0,24.70,2.5,0.00,11.466667,2023,3,2023/03_3
4,2,2023-03-01 00:08:04,2023-03-01 00:11:06,1.0,1.23,1.0,N,79,137,1,...,2.44,0.0,1.0,14.64,2.5,0.00,3.033333,2023,3,2023/03_4


In [ ]:
#Let's save the results and the ids
df_result = pd.DataFrame()
df_result['ride_id'] = df['ride_id']
df_result['tpep_pickup_datetime'] = df['tpep_pickup_datetime']
df_result['PULocationID'] = df['PULocationID']
df_result['DOLocationID'] = df['DOLocationID']
df_result['actual_duration'] = df['duration']
df_result['predicted_duration'] = y_pred


,ride_id,tpep_pickup_datetime,PULocationID,DOLocationID,actual_duration,predicted_duration
0,2023/03_0,2023-03-01 00:06:43,238,42,10.000000,16.245906
1,2023/03_1,2023-03-01 00:08:25,138,231,31.083333,26.134796
2,2023/03_2,2023-03-01 00:15:04,140,186,14.366667,11.884264
3,2023/03_3,2023-03-01 00:49:37,140,43,11.466667,11.997720
4,2023/03_4,2023-03-01 00:08:04,79,137,3.033333,10.234486


In [25]:
output_dir = "data_predictions"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "_results_yellow_tripdata_2023-03.parquet")
df_result.drop(columns=['tpep_pickup_datetime', 'PULocationID', 'DOLocationID',"actual_duration"]).to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
) #The size of the file is 66M

In [26]:
#The command to copy this file  is: cp ..//web-service-mlflow/nameofthemodel.ipynb score.ipynb
#jupyter nbconvert --to script score.ipynb
!jupyter nbconvert --to script scoring.ipynb


[NbConvertApp] WARNING | Config option `kernel_spec_manager_class` not recognized by `NbConvertApp`.
[NbConvertApp] Converting notebook scoring.ipynb to script
[NbConvertApp] Writing 1125 bytes to scoring.py
